# R Master v8b-Lite · Lap Head Beauty Preview

基线：**v8a Fix2 Head Frame Alignment**。

v8a Fix2 已经证明 Lap 头能正确挂到 Mona 身体上。v8b-Lite 只做 **3 张头肩审美图**，避开手机 Colab 在全身 8 图 Eevee 渲染时反复断线。

### v8b 做什么
- 保留 v8a Fix2 的头部位置 / 三轴对齐结果；
- 去掉 donor 的 Ear Outline 预览壳，避免重叠边线；
- 从用户自己的 Lapine 包提取：
  - Face_Tex_Color.tga
  - Hair_Tex_Color.tga
  - Other_Tex_Color.tga
- 给 Face / Eye / Eyebrow / Hair / Ear 重建轻量 Blender 材质；
- 头发 alpha 参与渲染，减少 Workbench 下“硬发片”错觉；
- 用 Eevee + 三点灯光重新渲染；
- Mona 身体只用中性材质，不抢头脸判断；
- 额外输出 donor mesh / material / shapekey 审计。

### 明确不做
- 不改 Mona 505 骨；
- 不转最终头部权重；
- 不焊 neck；
- 不烘焙 Rest Pose；
- 不导最终 VRM；
- 不碰 production。


In [ ]:
from google.colab import drive, files
from pathlib import Path
import shutil, subprocess, zipfile, tarfile, json, os, hashlib

BUILD_TAG="v8b_head_beauty_20260920_r1"
print("R Master v8b · Lap Head Beauty Preview")
drive.mount("/content/drive")

ROOT=Path("/content/drive/MyDrive/R_Master")
SRC=ROOT/"v8a_head_graft"/"fix2_latest"/"R_Master_v8a_Fix2_HEAD_GRAFT_PREVIEW.blend"
REF=ROOT/"reference"
CACHE=ROOT/"cache"
OUT=ROOT/"v8b_head_beauty"/"latest"
OUT.mkdir(parents=True,exist_ok=True)

LAPZIP=REF/"Lapine_Ver.1.11_A.zip"
if not SRC.exists() or SRC.stat().st_size<50*1024*1024:
    raise RuntimeError("没找到 v8a Fix2 preview。截图给二蛋。")
if not LAPZIP.exists() or LAPZIP.stat().st_size<70*1024*1024:
    raise RuntimeError("没找到 Drive 里的 Lapine_Ver.1.11_A.zip")
print(f"✓ Fix2 source：{SRC.stat().st_size/1024/1024:.1f} MiB")
print(f"✓ Lapine cache：{LAPZIP.stat().st_size/1024/1024:.1f} MiB")

h=hashlib.sha256()
with LAPZIP.open("rb") as f:
    for b in iter(lambda:f.read(8*1024*1024),b""): h.update(b)
sha=h.hexdigest()
EXPECTED="8e215aeae8d2e42719305d66292b84c2714f616b7952591dee0383c376c86af4"
print("Lapine SHA256:",sha)
if sha!=EXPECTED: raise RuntimeError("Lapine 源校验失败")
print("✓ Lapine 源校验通过")



In [ ]:
BLENDER_VERSION="4.4.3"
BLENDER_URL="https://download.blender.org/release/Blender4.4/blender-4.4.3-linux-x64.tar.xz"
LOCAL=Path("/content/r_master_v8b"); LOCAL.mkdir(parents=True,exist_ok=True)
ARCHIVE=LOCAL/f"blender-{BLENDER_VERSION}-linux-x64.tar.xz"
BDIR=LOCAL/f"blender-{BLENDER_VERSION}-linux-x64"
DRIVE_ARCHIVE=CACHE/ARCHIVE.name

if shutil.which("xvfb-run") is None:
    subprocess.run(["apt-get","update","-qq"],check=True,stdout=subprocess.DEVNULL,stderr=subprocess.STDOUT)
    subprocess.run(["apt-get","install","-y","-qq","xvfb","libgl1","libx11-6","libxi6","libxrender1","libxfixes3","libxkbcommon0","libsm6"],check=True,stdout=subprocess.DEVNULL,stderr=subprocess.STDOUT)

if DRIVE_ARCHIVE.exists() and DRIVE_ARCHIVE.stat().st_size>100*1024*1024:
    shutil.copy2(DRIVE_ARCHIVE,ARCHIVE)
    print("✓ 复用 Blender 缓存")
else:
    subprocess.run(["wget","-q","--show-progress","-O",str(ARCHIVE),BLENDER_URL],check=True)
    shutil.copy2(ARCHIVE,DRIVE_ARCHIVE)

if not (BDIR/"blender").exists():
    if BDIR.exists(): shutil.rmtree(BDIR)
    subprocess.run(["tar","-xf",str(ARCHIVE),"-C",str(LOCAL)],check=True)
BLENDER=BDIR/"blender"
print("✓ Blender ready")

# 只提取 v8b 需要的 3 张纹理。
TEXDIR=LOCAL/"textures"
if TEXDIR.exists(): shutil.rmtree(TEXDIR)
TEXDIR.mkdir()
outer=LOCAL/"lapine_outer"
if outer.exists(): shutil.rmtree(outer)
outer.mkdir()
with zipfile.ZipFile(LAPZIP,"r") as z:
    unity=[n for n in z.namelist() if n.lower().endswith("lapine.unitypackage")]
    if not unity: raise RuntimeError("Lapine.unitypackage missing")
    z.extract(unity[0],outer)
UNITY=outer/unity[0]

pkg=LOCAL/"lapine_pkg"
if pkg.exists(): shutil.rmtree(pkg)
pkg.mkdir()
with tarfile.open(UNITY,"r:*") as t: t.extractall(pkg)

wanted={
 "Assets/Models/Textures/Face_Tex_Color.tga":"Face_Tex_Color.tga",
 "Assets/Models/Textures/Hair_Tex_Color.tga":"Hair_Tex_Color.tga",
 "Assets/Models/Textures/Other_Tex_Color.tga":"Other_Tex_Color.tga",
}
found={}
for p in pkg.glob("*/pathname"):
    try:q=p.read_text(encoding="utf-8").strip()
    except:continue
    if q in wanted:
        a=p.parent/"asset"
        if a.exists():
            dst=TEXDIR/wanted[q]
            shutil.copy2(a,dst); found[q]=str(dst)
if len(found)!=len(wanted):
    raise RuntimeError("纹理提取不完整："+str(found))
print("✓ v8b textures extracted")



In [ ]:
BUILD=LOCAL/"R_Master_v8b_Build.py"
BUILD.write_text("\nimport bpy,sys,os,json,math\nfrom mathutils import Vector\n\nargv=sys.argv[sys.argv.index(\"--\")+1:] if \"--\" in sys.argv else []\nout=texdir=tag=None\nfor i,a in enumerate(argv):\n    if a==\"--out\":out=argv[i+1]\n    if a==\"--texdir\":texdir=argv[i+1]\n    if a==\"--tag\":tag=argv[i+1]\nif not out or not texdir: raise RuntimeError(\"args\")\nos.makedirs(out,exist_ok=True)\n\nbody=bpy.data.objects.get(\"R2_Mona_Main\")\nrig=bpy.data.objects.get(\"R_Master_Align_v2_PREVIEW\") or bpy.data.objects.get(\"Mona_Armature\")\ndonors=[o for o in bpy.data.objects if o.type==\"MESH\" and o.get(\"R_DONOR\")==\"LapineHead_v8a_fix2\"]\nif not body or not rig or not donors: raise RuntimeError(\"Fix2 preview objects missing\")\n\n# 1) 清掉只用于 toon outline 的 donor 壳，避免审美预览出现双层耳轮廓。\nremoved=[]\nfor o in list(donors):\n    if \"outline\" in o.name.lower():\n        removed.append(o.name)\n        bpy.data.objects.remove(o,do_unlink=True)\ndonors=[o for o in bpy.data.objects if o.type==\"MESH\" and o.get(\"R_DONOR\")==\"LapineHead_v8a_fix2\"]\n\n# 2) 读取真实 Lapine 颜色贴图。\nface_img=bpy.data.images.load(os.path.join(texdir,\"Face_Tex_Color.tga\"),check_existing=True)\nhair_img=bpy.data.images.load(os.path.join(texdir,\"Hair_Tex_Color.tga\"),check_existing=True)\nother_img=bpy.data.images.load(os.path.join(texdir,\"Other_Tex_Color.tga\"),check_existing=True)\n\ndef mk_tex_mat(name,img,rough=.62,alpha=True,emission=0.0):\n    m=bpy.data.materials.get(name) or bpy.data.materials.new(name)\n    m.use_nodes=True\n    nt=m.node_tree; nt.nodes.clear()\n    outn=nt.nodes.new(\"ShaderNodeOutputMaterial\")\n    bs=nt.nodes.new(\"ShaderNodeBsdfPrincipled\")\n    tx=nt.nodes.new(\"ShaderNodeTexImage\"); tx.image=img; tx.interpolation=\"Linear\"\n    bs.inputs[\"Roughness\"].default_value=rough\n    if \"Specular IOR Level\" in bs.inputs: bs.inputs[\"Specular IOR Level\"].default_value=.28\n    nt.links.new(tx.outputs[\"Color\"],bs.inputs[\"Base Color\"])\n    if alpha and \"Alpha\" in tx.outputs:\n        nt.links.new(tx.outputs[\"Alpha\"],bs.inputs[\"Alpha\"])\n        try:m.surface_render_method=\"DITHERED\"\n        except:pass\n    nt.links.new(bs.outputs[\"BSDF\"],outn.inputs[\"Surface\"])\n    return m\n\ndef mk_plain(name,rgba,rough=.75):\n    m=bpy.data.materials.get(name) or bpy.data.materials.new(name)\n    m.use_nodes=True\n    bs=m.node_tree.nodes.get(\"Principled BSDF\")\n    bs.inputs[\"Base Color\"].default_value=rgba\n    bs.inputs[\"Roughness\"].default_value=rough\n    if \"Specular IOR Level\" in bs.inputs: bs.inputs[\"Specular IOR Level\"].default_value=.22\n    return m\n\nM_FACE=mk_tex_mat(\"R_v8b_Lap_Face\",face_img,.68,True)\nM_HAIR=mk_tex_mat(\"R_v8b_Lap_Hair\",hair_img,.58,True)\nM_OTHER=mk_tex_mat(\"R_v8b_Lap_Other\",other_img,.60,True)\nM_BODY=mk_plain(\"R_v8b_Mona_Neutral\",(0.42,0.34,0.31,1),.82)\n\n# 单 atlas 审美预览：Face / Eyes / Brows 共用 Face atlas；Hair 单独；Ear 用 Other。\nfor o in donors:\n    n=o.name.lower()\n    if \"hair\" in n: mat=M_HAIR\n    elif \"ear\" in n: mat=M_OTHER\n    else: mat=M_FACE\n    o.data.materials.clear()\n    o.data.materials.append(mat)\n\n# Mona 身体只用中性材料；不改变 mesh / rig。\nbody.data.materials.clear()\nbody.data.materials.append(M_BODY)\n\n# 3) 轻微几何显示清理，不改变 donor 对齐：\n#    smooth shading + auto smooth-like normal behavior.\nfor o in donors+[body]:\n    for p in o.data.polygons: p.use_smooth=True\n\n# 4) 审计 donor 当前状态。\naudit=[]\nfor o in donors:\n    mats=[m.name if m else None for m in o.data.materials]\n    sk=[]\n    if o.data.shape_keys:\n        sk=[k.name for k in o.data.shape_keys.key_blocks]\n    audit.append({\n      \"object\":o.name,\n      \"verts\":len(o.data.vertices),\n      \"faces\":len(o.data.polygons),\n      \"materials\":mats,\n      \"shape_key_count\":len(sk),\n      \"shape_keys_sample\":sk[:30]\n    })\n\nreport={\n \"ok\":True,\n \"stage\":\"R_Master_v8b_LapHeadBeautyPreview\",\n \"build_tag\":tag,\n \"source\":\"v8a Fix2\",\n \"mona_bone_count\":len(rig.data.bones),\n \"donor_mesh_count\":len(donors),\n \"removed_preview_meshes\":removed,\n \"donor_audit\":audit,\n \"texture_sources\":[face_img.name,hair_img.name,other_img.name],\n \"alignment_changed\":False,\n \"weights_transferred\":False,\n \"neck_welded\":False,\n \"rest_pose_baked\":False,\n \"final_vrm\":False,\n \"note\":\"Beauty/material preview only; original Lapine texture atlases applied to donor objects.\"\n}\nwith open(os.path.join(out,\"R_Master_v8b_report.json\"),\"w\",encoding=\"utf-8\") as f:\n    json.dump(report,f,ensure_ascii=False,indent=2)\n\nbpy.ops.wm.save_as_mainfile(filepath=os.path.join(out,\"R_Master_v8b_HEAD_BEAUTY_PREVIEW.blend\"),check_existing=False)\nprint(\"[v8b] BUILD_OK\",len(donors),removed)\n",encoding="utf-8")
STAGE=OUT/"R_Master_v8b_HEAD_BEAUTY_PREVIEW.blend"
REPORT=OUT/"R_Master_v8b_report.json"

need=True
if STAGE.exists() and STAGE.stat().st_size>50*1024*1024 and REPORT.exists():
    try: need=json.loads(REPORT.read_text(encoding="utf-8")).get("build_tag")!=BUILD_TAG
    except: need=True

if need:
    TMP=LOCAL/"stage"
    if TMP.exists(): shutil.rmtree(TMP)
    TMP.mkdir()
    log=TMP/"R_Master_v8b_build.log"
    cmd=["xvfb-run","-a",str(BLENDER),"--background",str(SRC),"--python",str(BUILD),
         "--","--out",str(TMP),"--texdir",str(TEXDIR),"--tag",BUILD_TAG]
    with log.open("w",encoding="utf-8") as f:
        p=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in p.stdout:
            f.write(line)
            if "[v8b]" in line or "Traceback" in line or "RuntimeError" in line: print(line.rstrip())
        rc=p.wait()
    if rc!=0:
        print(log.read_text(encoding="utf-8",errors="replace")[-12000:])
        raise RuntimeError("v8b build failed")
    for n in ["R_Master_v8b_HEAD_BEAUTY_PREVIEW.blend","R_Master_v8b_report.json","R_Master_v8b_build.log"]:
        shutil.copy2(TMP/n,OUT/n)
else:
    print("✓ v8b build checkpoint exists")
print(json.dumps(json.loads(REPORT.read_text(encoding="utf-8")),ensure_ascii=False,indent=2)[:12000])



In [ ]:
RENDER=LOCAL/"R_Master_v8b_Render.py"
RENDER.write_text("\nimport bpy,os,sys,math\nfrom mathutils import Vector\nargv=sys.argv[sys.argv.index(\"--\")+1:] if \"--\" in sys.argv else []\nout=view=None\nfor i,a in enumerate(argv):\n    if a==\"--out\":out=argv[i+1]\n    if a==\"--view\":view=argv[i+1]\nbody=bpy.data.objects.get(\"R2_Mona_Main\")\nrig=bpy.data.objects.get(\"R_Master_Align_v2_PREVIEW\") or bpy.data.objects.get(\"Mona_Armature\")\ndonors=[o for o in bpy.data.objects if o.type==\"MESH\" and o.get(\"R_DONOR\")==\"LapineHead_v8a_fix2\"]\nif not body or not rig or not donors: raise RuntimeError(\"preview missing\")\n\nfor o in bpy.context.scene.objects:\n    if o.type==\"ARMATURE\":o.hide_render=True\n    if o.type==\"MESH\":o.hide_render=(o!=body and o not in donors)\n\nscene=bpy.context.scene\nscene.render.engine=\"BLENDER_EEVEE_NEXT\"\nscene.render.image_settings.file_format=\"PNG\"\nscene.render.film_transparent=False\nscene.render.resolution_percentage=100\nscene.world.color=(0.025,0.025,0.032)\n\n# Clean lights every isolated render process.\nfor o in list(bpy.data.objects):\n    if o.type==\"LIGHT\":bpy.data.objects.remove(o,do_unlink=True)\n\ndef add_area(name,loc,energy,size,color=(1,1,1)):\n    d=bpy.data.lights.new(name,\"AREA\"); d.energy=energy; d.shape=\"DISK\"; d.size=size; d.color=color\n    o=bpy.data.objects.new(name,d); scene.collection.objects.link(o); o.location=Vector(loc)\n    return o\n\nbp=[body.matrix_world@Vector(c) for c in body.bound_box]\nmn=Vector((min(p.x for p in bp),min(p.y for p in bp),min(p.z for p in bp)))\nmx=Vector((max(p.x for p in bp),max(p.y for p in bp),max(p.z for p in bp)))\nbc=(mn+mx)*.5; h=mx.z-mn.z; d=h*2.7\n\nhb=rig.data.bones.get(\"Head\")\nHC=rig.matrix_world@hb.head_local if hb else Vector((bc.x,bc.y,mx.z-h*.12))\nhead_target=HC+Vector((0,0,h*.075))\n\nkey=add_area(\"Key\",(bc.x-d*.55,bc.y-d*.65,bc.z+h*.25),1050,h*.95,(1.0,.88,.80))\nfill=add_area(\"Fill\",(bc.x+d*.60,bc.y-d*.28,bc.z+h*.12),700,h*.85,(.72,.82,1.0))\nrim=add_area(\"Rim\",(bc.x,bc.y+d*.65,bc.z+h*.30),900,h*.70,(.90,.95,1.0))\ndef aim(o,t):o.rotation_euler=(Vector(t)-o.location).to_track_quat(\"-Z\",\"Y\").to_euler()\nfor L in (key,fill,rim):aim(L,head_target if \"head\" in view else bc)\n\ncd=bpy.data.cameras.new(\"R_v8b_cam_data\")\ncam=bpy.data.objects.new(\"R_v8b_cam\",cd);scene.collection.objects.link(cam);scene.camera=cam\ncam.data.type=\"ORTHO\";cam.data.lens=70\ndef look(t):cam.rotation_euler=(Vector(t)-cam.location).to_track_quat(\"-Z\",\"Y\").to_euler()\ndef rr(fn,pos,tgt,scale,res):\n    scene.render.resolution_x,scene.render.resolution_y=res\n    cam.location=Vector(pos);cam.data.ortho_scale=scale;look(tgt)\n    scene.render.filepath=os.path.join(out,fn)\n    bpy.ops.render.render(write_still=True)\n\nspec={\n\"front\":(\"R_Master_v8b_front.png\",(bc.x,bc.y-d,bc.z),bc,h*1.07,(720,960)),\n\"side\":(\"R_Master_v8b_side.png\",(bc.x+d,bc.y,bc.z),bc,h*1.07,(720,960)),\n\"back\":(\"R_Master_v8b_back.png\",(bc.x,bc.y+d,bc.z),bc,h*1.07,(720,960)),\n\"three_quarter\":(\"R_Master_v8b_three_quarter.png\",(bc.x+d*.72,bc.y-d*.72,bc.z),bc,h*1.07,(720,960)),\n\"head_front\":(\"R_Master_v8b_head_front.png\",(HC.x,HC.y-d,head_target.z),head_target,h*.30,(640,640)),\n\"head_side\":(\"R_Master_v8b_head_side.png\",(HC.x+d,HC.y,head_target.z),head_target,h*.30,(640,640)),\n\"head_three_quarter\":(\"R_Master_v8b_head_three_quarter.png\",(HC.x+d*.72,HC.y-d*.72,head_target.z),head_target,h*.30,(640,640)),\n\"neck_interface\":(\"R_Master_v8b_neck_interface.png\",(HC.x+d*.70,HC.y-d*.70,HC.z-h*.015),HC+Vector((0,0,-h*.015)),h*.23,(900,720))\n}\nrr(*spec[view])\nprint(\"[v8b] RENDER_OK\",view)\n",encoding="utf-8")
views=[
("head_front","R_Master_v8b_head_front.png"),
("head_side","R_Master_v8b_head_side.png"),
("head_three_quarter","R_Master_v8b_head_three_quarter.png")]
for i,(v,f) in enumerate(views,1):
    p=OUT/f
    if p.exists() and p.stat().st_size>20000:
        print(f"✓ [{i}/8] {v} checkpoint");continue
    print(f"[{i}/8] rendering {v}")
    r=subprocess.run(["xvfb-run","-a",str(BLENDER),"--background",str(STAGE),"--python",str(RENDER),
                      "--","--out",str(OUT),"--view",v],
                     stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
    if r.returncode!=0:
        print(r.stdout[-12000:]);raise RuntimeError(v+" render failed")
print("✓ v8b-Lite 3 head renders complete")



In [ ]:
from IPython.display import display,Image,Markdown
items=[
("头肩正面","R_Master_v8b_head_front.png"),
("头肩侧面","R_Master_v8b_head_side.png"),
("头肩 3/4","R_Master_v8b_head_three_quarter.png")]
for title,f in items:
    display(Markdown("### "+title));display(Image(filename=str(OUT/f),width=500))

z=OUT/"R_Master_v8b_Lite_Review.zip"
if z.exists():z.unlink()
with zipfile.ZipFile(z,"w",compression=zipfile.ZIP_DEFLATED,compresslevel=6) as w:
    for _,f in items:w.write(OUT/f,arcname=f)
    for f in ("R_Master_v8b_report.json","R_Master_v8b_build.log"):
        if (OUT/f).exists():w.write(OUT/f,arcname=f)
print(f"✓ Review ZIP：{z.stat().st_size/1024/1024:.1f} MiB")
files.download(str(z))

